Import Library

In [1]:
import os
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

Path Dataset

In [2]:
RAW_PATH = Path("../data/raw")
OUTPUT_PATH = Path("../data/processed")

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

Helper Function

In [3]:
import re
import numpy as np

def clean_number(value):
    """
    Convert Indonesian marketplace numbers.

    Examples:
    ----------
    107.000   -> 107000
    Rp107.000 -> 107000
    596,4k    -> 596400
    1,6m      -> 1600000
    1RB+      -> 1000
    2,5RB+    -> 2500
    15RB      -> 15000
    1JT+      -> 1000000
    2,5JT     -> 2500000
    100%      -> 100
    """

    if value is None:
        return np.nan

    if isinstance(value, (int, float)):
        return value

    value = str(value).strip().lower()

    if value == "":
        return np.nan

    value = value.replace("rp", "")
    value = value.replace("%", "")
    value = value.replace("+", "")
    value = value.replace(" ", "")

    # Shopee Indonesia
    if "rb" in value:
        number = value.replace("rb", "").replace(",", ".")
        return float(number) * 1000

    if "jt" in value:
        number = value.replace("jt", "").replace(",", ".")
        return float(number) * 1000000

    # Format internasional
    if "k" in value:
        number = value.replace("k", "").replace(",", ".")
        return float(number) * 1000

    if "m" in value:
        number = value.replace("m", "").replace(",", ".")
        return float(number) * 1000000

    # Harga Indonesia
    value = value.replace(".", "")
    value = value.replace(",", ".")

    try:
        return float(value)

    except:
        return np.nan

Seller Joined

In [4]:
def extract_joined_year(value):

    if value is None:
        return np.nan

    match = re.search(r'(\d+)', str(value))

    if match:
        return int(match.group(1))

    return np.nan

Response Time

In [5]:
def encode_response_time(value):

    if value is None:
        return np.nan

    value = value.lower()

    if "minutes" in value:
        return 1

    if "hours" in value:
        return 2

    if "days" in value:
        return 3

    return np.nan

Read All JSON

In [6]:
records = []

for file in RAW_PATH.glob("*.json"):

    keyword = file.stem

    print(f"Loading {file.name}")

    with open(file, "r", encoding="utf-8") as f:
        raw = json.load(f)

    for item in raw["data"]:

        record = {

            "keyword": keyword,

            "product_id": item.get("id"),

            "product_name": item.get("name"),

            "url": item.get("url"),

            "currency": item["price"].get("currency"),

            "original_price_min": item["price"].get("original_price_min"),

            "original_price_max": item["price"].get("original_price_max"),

            "final_price_min": item["price"].get("final_price_min"),

            "final_price_max": item["price"].get("final_price_max"),

            "discount_percent": item["price"].get("discount_percent"),

            "category_1": item["categoryPath"][1] if len(item["categoryPath"]) > 1 else None,

            "category_2": item["categoryPath"][2] if len(item["categoryPath"]) > 2 else None,

            "category_3": item["categoryPath"][3] if len(item["categoryPath"]) > 3 else None,

            "rating": item["rating"].get("average"),

            "review_count": item["rating"].get("reviewCount"),

            "sold": item["rating"].get("sold"),

            "star_1": item["rating"]["starRating"].get("1"),

            "star_2": item["rating"]["starRating"].get("2"),

            "star_3": item["rating"]["starRating"].get("3"),

            "star_4": item["rating"]["starRating"].get("4"),

            "star_5": item["rating"]["starRating"].get("5"),

            "seller_id": item["seller"].get("id"),

            "seller_name": item["seller"].get("name"),

            "seller_rating": item["seller"].get("rating"),

            "seller_followers": item["seller"].get("follower"),

            "seller_product": item["seller"].get("product"),

            "seller_response_rate": item["seller"].get("responseRate"),

            "seller_response_time": item["seller"].get("responseTime"),

            "seller_joined": item["seller"].get("joined"),

        }

        records.append(record)

df = pd.DataFrame(records)

Loading headset.json
Loading laptop.json
Loading smartphone.json
Loading smartwatch.json
Loading tablet.json


Preview Dataset

In [7]:
print(df.shape)

df.head()

(250, 29)


,keyword,product_id,product_name,url,currency,original_price_min,original_price_max,final_price_min,final_price_max,discount_percent,...,star_4,star_5,seller_id,seller_name,seller_rating,seller_followers,seller_product,seller_response_rate,seller_response_time,seller_joined
0,headset,42531428335,VIVAN Earphone Headphone Headset In-Ear 3D Sub...,https://shopee.co.id/VIVAN-Earphone-Headphone-...,IDR,107.000,107.000,107.000,107.000,0,...,1,12,11722931,Vivan Official Shop,"596,4k","716,7k",477,100%,within minutes,10 years ago
1,headset,25117212844,Lenovo Thinkplus TH30 Headset Bluetooth V5.3 W...,https://shopee.co.id/Lenovo-Thinkplus-TH30-Hea...,IDR,240.000,480.000,142.000,289.000,53,...,654,"9,4k",1089578754,Thinkplus Audio Store,"56,8k","31,4k",235,100%,within hours,33 months ago
2,headset,826361914,Xiaomi In-Ear Headphones Basic | Built-in Micr...,https://shopee.co.id/Xiaomi-In-Ear-Headphones-...,IDR,119.000,119.000,99.000,99.000,17,...,"4,4k","57,8k",51925611,Xiaomi Official Store,"3,2m","3,4m",341,95%,within hours,8 years ago
3,headset,40651424918,Pods Air Headset EarPods Kabel Earphone Handsf...,https://shopee.co.id/Pods-Air-Headset-EarPods-...,IDR,185.165,185.165,185.165,185.165,0,...,1,28,1453081456,Pods Air,"4,1k","3,1k",12,79%,within hours,18 months ago
4,headset,43662379094,Baseus Bass BH1 Lite Wireless Headphone Super ...,https://shopee.co.id/Baseus-Bass-BH1-Lite-Wire...,IDR,371.420,371.420,371.420,371.420,0,...,25,715,223032375,Baseus Official Shop,"379,9k","249,8k",262,100%,within minutes,6 years ago


Cleaning Numeric Data

In [8]:
numeric_columns = [

    "original_price_min",

    "original_price_max",

    "final_price_min",

    "final_price_max",

    "discount_percent",

    "rating",

    "review_count",

    "sold",

    "star_1",

    "star_2",

    "star_3",

    "star_4",

    "star_5",

    "seller_rating",

    "seller_followers",

    "seller_product",

    "seller_response_rate"

]

for col in numeric_columns:
    df[col] = df[col].apply(clean_number)

Feature Engineering

In [9]:
df["price_avg"] = (
    df["final_price_min"] +
    df["final_price_max"]
) / 2

df["price_range"] = (
    df["final_price_max"] -
    df["final_price_min"]
)

df["seller_age"] = df["seller_joined"].apply(extract_joined_year)

df["seller_response_time"] = df["seller_response_time"].apply(
    encode_response_time
)

Missing Value

In [10]:
missing = pd.DataFrame({

    "Missing": df.isnull().sum(),

    "Percent": (df.isnull().sum()/len(df))*100

})

missing.sort_values("Percent", ascending=False)

,Missing,Percent
rating,5,2.0
keyword,0,0.0
product_name,0,0.0
product_id,0,0.0
currency,0,0.0
original_price_min,0,0.0
original_price_max,0,0.0
url,0,0.0
final_price_min,0,0.0
final_price_max,0,0.0


Duplicate

In [11]:
print("Duplicate :", df.duplicated().sum())

df = df.drop_duplicates()

Duplicate : 0


Data Info

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 32 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   keyword               250 non-null    str    
 1   product_id            250 non-null    str    
 2   product_name          250 non-null    str    
 3   url                   250 non-null    str    
 4   currency              250 non-null    str    
 5   original_price_min    250 non-null    float64
 6   original_price_max    250 non-null    float64
 7   final_price_min       250 non-null    float64
 8   final_price_max       250 non-null    float64
 9   discount_percent      250 non-null    int64  
 10  category_1            250 non-null    str    
 11  category_2            250 non-null    str    
 12  category_3            250 non-null    str    
 13  rating                245 non-null    float64
 14  review_count          250 non-null    float64
 15  sold                  250 non-null

Statistik

In [13]:
df.describe(include="all")

,keyword,product_id,product_name,url,currency,original_price_min,original_price_max,final_price_min,final_price_max,discount_percent,...,seller_name,seller_rating,seller_followers,seller_product,seller_response_rate,seller_response_time,seller_joined,price_avg,price_range,seller_age
count,250,250,250,250,250,2.500000e+02,2.500000e+02,2.500000e+02,2.500000e+02,250.000000,...,250,2.500000e+02,2.500000e+02,250.000000,250.00000,250.000000,250,2.500000e+02,2.500000e+02,250.000000
unique,5,249,245,250,1,NaN,NaN,NaN,NaN,NaN,...,181,NaN,NaN,NaN,NaN,NaN,35,NaN,NaN,NaN
top,headset,40303069744,Headset Premium Stereo Sound JETE HX12 - Garan...,https://shopee.co.id/VIVAN-Earphone-Headphone-...,IDR,NaN,NaN,NaN,NaN,NaN,...,Advan Notebook Official Store,NaN,NaN,NaN,NaN,NaN,6 years ago,NaN,NaN,NaN
freq,50,2,2,1,250,NaN,NaN,NaN,NaN,NaN,...,5,NaN,NaN,NaN,NaN,NaN,42,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,4.264358e+06,4.549178e+06,4.066316e+06,4.340786e+06,12.900000,...,NaN,1.813529e+05,2.679919e+05,413.080000,97.78800,1.468000,NaN,4.203551e+06,2.744699e+05,10.344000
std,NaN,NaN,NaN,NaN,NaN,4.680755e+06,5.023062e+06,4.649063e+06,4.999645e+06,23.601315,...,NaN,4.855449e+05,7.322273e+05,640.476869,5.58748,0.499976,NaN,4.805044e+06,9.309669e+05,7.851129
min,NaN,NaN,NaN,NaN,NaN,2.200000e+04,2.200000e+04,1.150000e+04,1.150000e+04,0.000000,...,NaN,5.300000e+01,1.600000e+01,1.000000,63.00000,1.000000,NaN,1.150000e+04,0.000000e+00,2.000000
25%,NaN,NaN,NaN,NaN,NaN,4.990000e+05,5.087500e+05,2.776700e+05,2.902500e+05,0.000000,...,NaN,4.475000e+03,5.300000e+03,57.000000,99.00000,1.000000,NaN,2.839350e+05,0.000000e+00,6.000000
50%,NaN,NaN,NaN,NaN,NaN,2.660000e+06,2.660000e+06,2.409000e+06,2.598500e+06,0.000000,...,NaN,1.770000e+04,2.195000e+04,194.500000,100.00000,1.000000,NaN,2.598500e+06,0.000000e+00,8.000000
75%,NaN,NaN,NaN,NaN,NaN,6.907750e+06,7.274751e+06,6.735725e+06,7.244000e+06,14.750000,...,NaN,7.595000e+04,9.070000e+04,396.500000,100.00000,2.000000,NaN,6.985999e+06,0.000000e+00,10.000000


Save CSV

In [14]:
output = OUTPUT_PATH / "ecommerce.csv"

df.to_csv(output, index=False, encoding="utf-8-sig")

print(output)

..\data\processed\ecommerce.csv


Final Preview

In [15]:
df.head()

,keyword,product_id,product_name,url,currency,original_price_min,original_price_max,final_price_min,final_price_max,discount_percent,...,seller_name,seller_rating,seller_followers,seller_product,seller_response_rate,seller_response_time,seller_joined,price_avg,price_range,seller_age
0,headset,42531428335,VIVAN Earphone Headphone Headset In-Ear 3D Sub...,https://shopee.co.id/VIVAN-Earphone-Headphone-...,IDR,107000.0,107000.0,107000.0,107000.0,0,...,Vivan Official Shop,596400.0,716700.0,477.0,100.0,1,10 years ago,107000.0,0.0,10
1,headset,25117212844,Lenovo Thinkplus TH30 Headset Bluetooth V5.3 W...,https://shopee.co.id/Lenovo-Thinkplus-TH30-Hea...,IDR,240000.0,480000.0,142000.0,289000.0,53,...,Thinkplus Audio Store,56800.0,31400.0,235.0,100.0,2,33 months ago,215500.0,147000.0,33
2,headset,826361914,Xiaomi In-Ear Headphones Basic | Built-in Micr...,https://shopee.co.id/Xiaomi-In-Ear-Headphones-...,IDR,119000.0,119000.0,99000.0,99000.0,17,...,Xiaomi Official Store,3200000.0,3400000.0,341.0,95.0,2,8 years ago,99000.0,0.0,8
3,headset,40651424918,Pods Air Headset EarPods Kabel Earphone Handsf...,https://shopee.co.id/Pods-Air-Headset-EarPods-...,IDR,185165.0,185165.0,185165.0,185165.0,0,...,Pods Air,4100.0,3100.0,12.0,79.0,2,18 months ago,185165.0,0.0,18
4,headset,43662379094,Baseus Bass BH1 Lite Wireless Headphone Super ...,https://shopee.co.id/Baseus-Bass-BH1-Lite-Wire...,IDR,371420.0,371420.0,371420.0,371420.0,0,...,Baseus Official Shop,379900.0,249800.0,262.0,100.0,1,6 years ago,371420.0,0.0,6
